# DAIR Pharmacy Access App — Precompute Script

Produces five static JSON files consumed by the scrollytelling app (N2, N4, N5 components).

| Output file | Description |
|---|---|
| `chart_n2_gauteng.json` | EA_TYPE aggregation, full Gauteng province |
| `chart_n2_kzn.json` | EA_TYPE aggregation, full KZN province |
| `chart_n4_ohb.json` | EA_TYPE aggregation, Olievenhoutbosch bbox |
| `chart_n4_kwamashu.json` | EA_TYPE aggregation, KwaMashu bbox |
| `chart_n5_access_pop.json` | Walk typology population tiers, both provinces combined |

**Implementation notes:**
- `_*` EA_TYPE variants (e.g. `Farms_*`) are stripped and merged with their base type
- Racial percentages use `sum(racial cols)` as denominator — `sal2023_est` is a scaled 2023 projection and does not match the raw census racial counts
- EA_TYPE rows with zero population/racial data are dropped (flagged/vacant parcels with no residents)
- Bbox filters use polygon centroids in WGS84, matching the JS `getCentroid()` behaviour

## 0 · Config

In [1]:
import os

GAUTENG_PATH = r"C:\Users\jcahi\Box\DAIR_SA\Data Sets\For Alex\gauteng_polygons.geojson"
KZN_PATH     = r"C:\Users\jcahi\Box\DAIR_SA\Data Sets\For Alex\kzn_polygons.geojson"
OUTPUT_DIR   = r"C:\Users\jcahi\Box\DAIR_SA\Data Sets\For Alex\dair data fix"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Output directory:", OUTPUT_DIR)
print("Gauteng input exists:", os.path.exists(GAUTENG_PATH))
print("KZN input exists:    ", os.path.exists(KZN_PATH))

Output directory: C:\Users\jcahi\Box\DAIR_SA\Data Sets\For Alex\dair data fix
Gauteng input exists: True
KZN input exists:     True


## 1 · Dependencies

In [ ]:
# Uncomment if geopandas is not already installed
# !pip install geopandas pyogrio

In [3]:
import json
import geopandas as gpd
import pandas as pd

print("geopandas", gpd.__version__)

c:\Users\jcahi\anaconda3\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\jcahi\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


geopandas 1.0.1


## 2 · Constants

In [4]:
CHART_EA_TYPES = {
    'Township', 'Informal residential', 'Formal residential', 'Suburb',
    'Traditional residential', 'Smallholdings', 'Small holdings',
    'Farms', 'Commercial', 'Industrial'
}

EA_TYPE_ORDER = [
    'Township', 'Informal residential', 'Formal residential', 'Suburb',
    'Traditional residential', 'Smallholdings', 'Small holdings',
    'Farms', 'Commercial', 'Industrial'
]

# \n characters are intentional — D3 splits on them for x-axis label wrapping
EA_TYPE_LABELS = {
    'Township':                'Township',
    'Informal residential':    'Informal\nresidential',
    'Formal residential':      'Formal\nresidential',
    'Suburb':                  'Suburb',
    'Traditional residential': 'Traditional\nresidential',
    'Smallholdings':           'Small-\nholdings',
    'Small holdings':          'Small\nholdings',
    'Farms':                   'Farms',
    'Commercial':              'Commercial',
    'Industrial':              'Industrial',
}

ACCESS_TIERS = [
    {'key': 'Well-served',         'color': '#27AE60'},
    {'key': 'Adequate',            'color': '#82C46C'},
    {'key': 'Underserved',         'color': '#F1C40F'},
    {'key': 'Demand overcrowding', 'color': '#E67E22'},
    {'key': 'Connectivity gap',    'color': '#C0392B'},
    {'key': 'Pharmacy desert',     'color': '#8B0000'},
]

# Bounding boxes: (minLng, minLat, maxLng, maxLat)
BBOX_OHB      = (27.94, -26.12, 28.24, -25.82)
BBOX_KWAMASHU = (30.81, -29.93, 31.11, -29.63)

print("Constants loaded.")

Constants loaded.


## 3 · Helper functions

In [5]:
def inspect_columns(gdf, label):
    """Print all column names and auto-resolve expected fields (handles shapefile truncation)."""
    cols = gdf.columns.tolist()
    print(f"\n{'='*60}")
    print(f"  {label} — {len(gdf):,} features")
    print(f"{'='*60}")
    print("All columns:", cols)

    expected_map = {
        'sal2023_est':   ['sal2023_est', 'sal2023_es'],
        'area_km2':      ['area_km2'],
        'Black_African': ['Black_African', 'Black_Afri'],
        'Coloured':      ['Coloured'],
        'Indian_Asian':  ['Indian_Asian', 'Indian_Asi'],
        'White':         ['White'],
        'Other':         ['Other'],
        'EA_TYPE':       ['EA_TYPE'],
        'walk_typology': ['walk_typology', 'walk_typol'],
    }
    print("\nField resolution:")
    resolved = {}
    for canonical, candidates in expected_map.items():
        found = next((c for c in candidates if c in cols), None)
        status = f"found as '{found}'" if found else "NOT FOUND"
        print(f"  {'ok' if found else '!!'} {canonical:20s} -> {status}")
        resolved[canonical] = found
    return resolved


def ensure_wgs84(gdf, label):
    """Reproject to WGS84 if needed so degree-based bounding boxes work correctly."""
    if gdf.crs is None:
        print(f"  !! {label}: CRS is None, assuming WGS84")
        return gdf
    if gdf.crs.to_epsg() == 4326:
        print(f"  ok {label}: already WGS84")
        return gdf
    print(f"  .. {label}: reprojecting {gdf.crs} -> EPSG:4326")
    return gdf.to_crs(epsg=4326)


def filter_by_bbox(gdf, bbox, label=""):
    """Keep features whose centroid falls within bbox (minLng, minLat, maxLng, maxLat)."""
    minLng, minLat, maxLng, maxLat = bbox
    centroids = gdf.geometry.centroid
    mask = (
        (centroids.x >= minLng) & (centroids.x <= maxLng) &
        (centroids.y >= minLat) & (centroids.y <= maxLat)
    )
    result = gdf[mask]
    print(f"  {label} bbox filter: {len(gdf):,} -> {len(result):,} features")
    if len(result) < 5:
        print(f"  !! WARNING: fewer than 5 features — bbox may be wrong or CRS mismatch")
    return result


def aggregate_ea_types(gdf, col_map, label=""):
    """
    Aggregate by EA_TYPE and return output-ready list in EA_TYPE_ORDER.

    Data quirks handled:
      1. _* suffix variants (e.g. 'Farms_*') are stripped and merged with base type.
      2. Racial percentages use sum(racial cols) as denominator, not sal2023_est.
         sal2023_est is a 2023 scaled projection that does not match raw census racial counts.
      3. Rows where race_total == 0 or population == 0 are dropped
         (these are flagged/vacant _* parcels with no residents).
    """
    ea_col   = col_map['EA_TYPE']
    pop_col  = col_map['sal2023_est']
    area_col = col_map['area_km2']
    ba_col   = col_map['Black_African']
    co_col   = col_map['Coloured']
    ia_col   = col_map['Indian_Asian']
    wh_col   = col_map['White']
    ot_col   = col_map['Other']

    missing = [k for k, v in col_map.items() if v is None and k != 'walk_typology']
    if missing:
        raise ValueError(f"{label}: missing required columns {missing} — fix col_map in cell 4b")

    df = gdf.copy()

    # Strip _* suffix so starred variants merge with their base type
    df[ea_col] = df[ea_col].str.replace(r'_\*$', '', regex=True).str.strip()

    # Warn about EA_TYPE values not in CHART_EA_TYPES and not known exclusions
    known_exclusions = {'Vacant', 'Collective living quarters', 'Parks and recreation'}
    all_types  = set(df[ea_col].dropna().unique())
    unexpected = all_types - CHART_EA_TYPES - known_exclusions
    if unexpected:
        print(f"  !! {label}: unexpected EA_TYPE values (check spellings): {sorted(unexpected)}")

    filtered = df[df[ea_col].isin(CHART_EA_TYPES)].copy()
    print(f"  {label}: {len(gdf):,} total -> {len(filtered):,} after EA_TYPE filter (incl. _* merge)")

    agg_cols = {
        pop_col:  'sum',
        area_col: 'sum',
        ba_col:   'sum',
        co_col:   'sum',
        ia_col:   'sum',
        wh_col:   'sum',
        ot_col:   'sum',
    }
    groups = filtered.groupby(ea_col).agg(agg_cols).reset_index()

    results = []
    for ea_type in EA_TYPE_ORDER:
        row = groups[groups[ea_col] == ea_type]
        if row.empty:
            continue
        r = row.iloc[0]

        race_total = r[ba_col] + r[co_col] + r[ia_col] + r[wh_col] + r[ot_col]

        # Drop rows with no population or racial data (flagged/vacant _* only parcels)
        if race_total == 0 or r[pop_col] == 0:
            print(f"  .. Skipping '{ea_type}' — zero population (likely _* only, no residents)")
            continue

        density = round(r[pop_col] / r[area_col]) if r[area_col] > 0 else 0

        results.append({
            'type':          EA_TYPE_LABELS[ea_type],
            'density':       int(density),
            'Black African': round((r[ba_col] / race_total) * 100),
            'Coloured':      round((r[co_col] / race_total) * 100),
            'Indian/Asian':  round((r[ia_col] / race_total) * 100),
            'White':         round((r[wh_col] / race_total) * 100),
            'Other':         round((r[ot_col] / race_total) * 100),
        })
    return results


def validate_ea_output(data, label):
    """Sanity-check EA_TYPE output: race sums ~100, density plausible."""
    print(f"\n-- Validation: {label} --")
    print(f"  EA_TYPEs in output: {[r['type'] for r in data]}")
    all_ok = True
    for r in data:
        race_sum = r['Black African'] + r['Coloured'] + r['Indian/Asian'] + r['White'] + r['Other']
        ok = abs(race_sum - 100) <= 2
        if not ok:
            all_ok = False
        flag = "ok" if ok else "!!"
        print(f"  {flag} {r['type'][:22]:22s}  density={r['density']:>7,}  race_sum={race_sum}")
    if all_ok:
        print("  All race sums within acceptable range (99-101).")


def write_json(data, filename, label):
    path = os.path.join(OUTPUT_DIR, filename)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"  ok Wrote {label} -> {path}  ({len(data)} rows)")


print("Helpers defined.")

Helpers defined.


## 4 · Load files and inspect columns

In [6]:
print("Loading Gauteng (~134 MB, may take a moment)...")
gdf_gauteng = gpd.read_file(GAUTENG_PATH)
col_map_gauteng = inspect_columns(gdf_gauteng, "gauteng_polygons.geojson")

print("\nLoading KZN...")
gdf_kzn = gpd.read_file(KZN_PATH)
col_map_kzn = inspect_columns(gdf_kzn, "kzn_polygons.geojson")

Loading Gauteng (~134 MB, may take a moment)...

  gauteng_polygons.geojson — 21,182 features
All columns: ['EA_CODE', 'EA_TYPE', 'White', 'Black_African', 'Coloured', 'Indian_Asian', 'Other', 'area_km2', 'sal2023_est', 'walk_typology', 'walk_dist_k1_km', 'exceeds_walk_k1_3km', 'econ_status', 'Ai_walk', 'walk_snap_flag', 'drive_typology', 'walk_dist_k3_km', 'PR_NAME', 'MN_NAME', 'SP_NAME', 'geometry']

Field resolution:
  ok sal2023_est          -> found as 'sal2023_est'
  ok area_km2             -> found as 'area_km2'
  ok Black_African        -> found as 'Black_African'
  ok Coloured             -> found as 'Coloured'
  ok Indian_Asian         -> found as 'Indian_Asian'
  ok White                -> found as 'White'
  ok Other                -> found as 'Other'
  ok EA_TYPE              -> found as 'EA_TYPE'
  ok walk_typology        -> found as 'walk_typology'

Loading KZN...

  kzn_polygons.geojson — 17,995 features
All columns: ['EA_CODE', 'EA_TYPE', 'White', 'Black_African', 'Colo

## 4b · Manual column override (only if needed)

If the cell above printed `!! ... NOT FOUND` for any required field, override it here.

```python
# Example:
# col_map_gauteng['Indian_Asian'] = 'Ind_Asian'
# col_map_kzn['walk_typology'] = 'wlk_typ'
```

If everything resolved cleanly, just run as-is.

In [7]:
# Add any overrides here if needed

print("Gauteng col_map:", {k: v for k, v in col_map_gauteng.items() if v})
print("KZN col_map:    ", {k: v for k, v in col_map_kzn.items() if v})

Gauteng col_map: {'sal2023_est': 'sal2023_est', 'area_km2': 'area_km2', 'Black_African': 'Black_African', 'Coloured': 'Coloured', 'Indian_Asian': 'Indian_Asian', 'White': 'White', 'Other': 'Other', 'EA_TYPE': 'EA_TYPE', 'walk_typology': 'walk_typology'}
KZN col_map:     {'sal2023_est': 'sal2023_est', 'area_km2': 'area_km2', 'Black_African': 'Black_African', 'Coloured': 'Coloured', 'Indian_Asian': 'Indian_Asian', 'White': 'White', 'Other': 'Other', 'EA_TYPE': 'EA_TYPE', 'walk_typology': 'walk_typology'}


## 5 · N2 — Full-province EA_TYPE aggregations

`chart_n2_gauteng.json` and `chart_n2_kzn.json`

In [8]:
print("=" * 60)
print("N2 GAUTENG")
print("=" * 60)
n2_gauteng = aggregate_ea_types(gdf_gauteng, col_map_gauteng, "N2 Gauteng")
validate_ea_output(n2_gauteng, "chart_n2_gauteng")
write_json(n2_gauteng, "chart_n2_gauteng.json", "chart_n2_gauteng")

print()
print("=" * 60)
print("N2 KZN")
print("=" * 60)
n2_kzn = aggregate_ea_types(gdf_kzn, col_map_kzn, "N2 KZN")
validate_ea_output(n2_kzn, "chart_n2_kzn")
write_json(n2_kzn, "chart_n2_kzn.json", "chart_n2_kzn")

N2 GAUTENG
  N2 Gauteng: 21,182 total -> 20,098 after EA_TYPE filter (incl. _* merge)
  .. Skipping 'Formal residential' — zero population (likely _* only, no residents)

-- Validation: chart_n2_gauteng --
  EA_TYPEs in output: ['Township', 'Informal\nresidential', 'Suburb', 'Traditional\nresidential', 'Small-\nholdings', 'Farms', 'Commercial', 'Industrial']
  ok Township                density=  7,295  race_sum=100
  ok Informal
residential    density=  6,787  race_sum=100
  ok Suburb                  density=  2,487  race_sum=100
  ok Traditional
residentia  density=  3,457  race_sum=99
  ok Small-
holdings         density=     91  race_sum=100
  ok Farms                   density=      9  race_sum=101
  ok Commercial              density=    879  race_sum=100
  ok Industrial              density=    102  race_sum=101
  All race sums within acceptable range (99-101).
  ok Wrote chart_n2_gauteng -> C:\Users\jcahi\Box\DAIR_SA\Data Sets\For Alex\dair data fix\chart_n2_gauteng.json  (8 r

## 6 · N4 — Bounding-box filtered EA_TYPE aggregations

`chart_n4_ohb.json` and `chart_n4_kwamashu.json`

Centroid-based bbox filter matching JS `getCentroid()` behaviour. Auto-reprojects to WGS84 if needed.

In [9]:
# Reproject to WGS84 so degree-based bboxes work regardless of source CRS
gdf_gauteng_wgs = ensure_wgs84(gdf_gauteng, "Gauteng")
gdf_kzn_wgs    = ensure_wgs84(gdf_kzn,     "KZN")

print()
print("=" * 60)
print("N4 OLIEVENHOUTBOSCH")
print("=" * 60)
gdf_ohb = filter_by_bbox(gdf_gauteng_wgs, BBOX_OHB, "OHB")
n4_ohb = aggregate_ea_types(gdf_ohb, col_map_gauteng, "N4 OHB")
validate_ea_output(n4_ohb, "chart_n4_ohb")
write_json(n4_ohb, "chart_n4_ohb.json", "chart_n4_ohb")

print()
print("=" * 60)
print("N4 KWAMASHU")
print("=" * 60)
gdf_kwm = filter_by_bbox(gdf_kzn_wgs, BBOX_KWAMASHU, "KwaMashu")
n4_kwm = aggregate_ea_types(gdf_kwm, col_map_kzn, "N4 KwaMashu")
validate_ea_output(n4_kwm, "chart_n4_kwamashu")
write_json(n4_kwm, "chart_n4_kwamashu.json", "chart_n4_kwamashu")

  ok Gauteng: already WGS84
  ok KZN: already WGS84

N4 OLIEVENHOUTBOSCH


C:\Users\jcahi\AppData\Local\Temp/ipykernel_18544/3195663801.py:45: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroids = gdf.geometry.centroid
C:\Users\jcahi\AppData\Local\Temp/ipykernel_18544/3195663801.py:45: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroids = gdf.geometry.centroid


  OHB bbox filter: 21,182 -> 3,338 features
  N4 OHB: 3,338 total -> 3,251 after EA_TYPE filter (incl. _* merge)
  .. Skipping 'Formal residential' — zero population (likely _* only, no residents)

-- Validation: chart_n4_ohb --
  EA_TYPEs in output: ['Township', 'Informal\nresidential', 'Suburb', 'Small-\nholdings', 'Farms', 'Commercial', 'Industrial']
  ok Township                density= 10,212  race_sum=100
  ok Informal
residential    density= 21,737  race_sum=100
  ok Suburb                  density=  1,970  race_sum=101
  ok Small-
holdings         density=    117  race_sum=101
  ok Farms                   density=    107  race_sum=100
  ok Commercial              density=    569  race_sum=99
  ok Industrial              density=    142  race_sum=100
  All race sums within acceptable range (99-101).
  ok Wrote chart_n4_ohb -> C:\Users\jcahi\Box\DAIR_SA\Data Sets\For Alex\dair data fix\chart_n4_ohb.json  (7 rows)

N4 KWAMASHU
  KwaMashu bbox filter: 17,995 -> 3,613 features
  N4 

## 7 · N5 — Walk typology population tiers (both provinces combined)

`chart_n5_access_pop.json`

In [10]:
print("=" * 60)
print("N5 WALK TYPOLOGY")
print("=" * 60)

wt_col_g  = col_map_gauteng.get('walk_typology')
wt_col_k  = col_map_kzn.get('walk_typology')
pop_col_g = col_map_gauteng['sal2023_est']
pop_col_k = col_map_kzn['sal2023_est']

if wt_col_g is None:
    raise ValueError("walk_typology not found in Gauteng — check col_map_gauteng in cell 4b")
if wt_col_k is None:
    raise ValueError("walk_typology not found in KZN — check col_map_kzn in cell 4b")

# Combine both provinces into one table
g_wt = gdf_gauteng[[wt_col_g, pop_col_g]].rename(
    columns={wt_col_g: 'walk_typology', pop_col_g: 'population'})
k_wt = gdf_kzn[[wt_col_k, pop_col_k]].rename(
    columns={wt_col_k: 'walk_typology', pop_col_k: 'population'})
combined = pd.concat([g_wt, k_wt], ignore_index=True)

# Flag any walk_typology values not in ACCESS_TIERS
all_tiers  = set(combined['walk_typology'].dropna().unique())
tier_keys  = {t['key'] for t in ACCESS_TIERS}
unexpected = all_tiers - tier_keys
if unexpected:
    print(f"!! walk_typology values not in ACCESS_TIERS: {sorted(unexpected)}")

tier_totals = combined.groupby('walk_typology')['population'].sum()
grand_total = combined['population'].sum()
print(f"Grand total population (both provinces): {grand_total:,.0f}")
print()

results_n5 = []
for tier in ACCESS_TIERS:
    key = tier['key']
    pop = tier_totals.get(key, 0)
    pct = round((pop / grand_total) * 1000) / 10 if grand_total > 0 else 0.0
    results_n5.append({
        'name':       key,
        'population': int(pop),
        'percent':    pct,
        'color':      tier['color'],
    })

# Validation
total_pct  = sum(r['percent'] for r in results_n5)
desert_gap = sum(r['percent'] for r in results_n5
                 if r['name'] in ('Pharmacy desert', 'Connectivity gap'))

for r in results_n5:
    print(f"  {r['name']:25s}  pop={r['population']:>12,}  pct={r['percent']:5.1f}%")

print(f"\n  Percent sum:                {total_pct:.1f}  (expect ~100.0)  {'ok' if abs(total_pct - 100) <= 0.2 else '!! off'}")
print(f"  Pharmacy desert + Conn gap: {desert_gap:.1f}%  (expect 70%+)   {'ok' if desert_gap >= 70 else '!! below expected threshold'}")

write_json(results_n5, "chart_n5_access_pop.json", "chart_n5_access_pop")

N5 WALK TYPOLOGY
!! walk_typology values not in ACCESS_TIERS: ['Access gap', 'Artifact zone']
Grand total population (both provinces): 28,092,560

  Well-served                pop=   6,623,965  pct= 23.6%
  Adequate                   pop=           0  pct=  0.0%
  Underserved                pop=           0  pct=  0.0%
  Demand overcrowding        pop=   5,383,357  pct= 19.2%
  Connectivity gap           pop=   2,169,585  pct=  7.7%
  Pharmacy desert            pop=  10,419,384  pct= 37.1%

  Percent sum:                87.6  (expect ~100.0)  !! off
  Pharmacy desert + Conn gap: 44.8%  (expect 70%+)   !! below expected threshold
  ok Wrote chart_n5_access_pop -> C:\Users\jcahi\Box\DAIR_SA\Data Sets\For Alex\dair data fix\chart_n5_access_pop.json  (6 rows)


## 8 · Summary

In [11]:
expected_files = [
    'chart_n2_gauteng.json',
    'chart_n2_kzn.json',
    'chart_n4_ohb.json',
    'chart_n4_kwamashu.json',
    'chart_n5_access_pop.json',
]

print("Output directory:", OUTPUT_DIR)
print()
all_present = True
for fname in expected_files:
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(fpath):
        size_kb = os.path.getsize(fpath) / 1024
        with open(fpath, encoding='utf-8') as f:
            data = json.load(f)
        print(f"  ok {fname:35s} {len(data):>3} rows  {size_kb:.1f} KB")
    else:
        print(f"  !! MISSING: {fname}")
        all_present = False

print()
if all_present:
    print("All five files present. Hand them back to Alex.")
else:
    print("Some files are missing — check cells above for errors.")

Output directory: C:\Users\jcahi\Box\DAIR_SA\Data Sets\For Alex\dair data fix

  ok chart_n2_gauteng.json                 8 rows  1.3 KB
  ok chart_n2_kzn.json                     8 rows  1.3 KB
  ok chart_n4_ohb.json                     7 rows  1.1 KB
  ok chart_n4_kwamashu.json                8 rows  1.3 KB
  ok chart_n5_access_pop.json              6 rows  0.7 KB

All five files present. Hand them back to Alex.
